# E-Commerce Platform — Analytical SQL & KPI Project

> **Star Schema Data Warehouse | DuckDB | Window Analytics | Recommendation System**

---

## 0. Setup & Imports

In [ ]:
# ── Install Dependencies (uncomment for Google Colab) ──
# !pip install duckdb faker plotly kaleido mlxtend scikit-learn -q

import duckdb
import numpy as np
import pandas as pd
import random
from datetime import date, timedelta
from faker import Faker

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# DuckDB connection (in-memory for speed; change to 'ecommerce.db' to persist)
conn = duckdb.connect(':memory:')

print("Setup complete — DuckDB version:", duckdb.__version__)

---
## 1. Schema Design + ERD

### 1.1 Star Schema ERD

```
                          ┌──────────────┐
                          │   Dim_Date   │
                          │──────────────│
                          │ date_key (PK)│
                          │ full_date    │
                          │ day          │
                          │ month        │
                          │ month_name   │
                          │ quarter      │
                          │ year         │
                          │ week_number  │
                          └──────┬───────┘
                                 │
┌─────────────────┐      ┌──────┴────────────────┐      ┌──────────────────┐
│  Dim_Customer   │      │   Fact_Order_Line     │      │   Dim_Product    │
│─────────────────│      │───────────────────────│      │──────────────────│
│customer_key (PK)│◄─────│ order_line_id (PK)    │─────►│product_key (PK)  │
│ customer_id     │      │ order_id              │      │ product_id       │
│ customer_name   │      │ date_key (FK)         │      │ product_name     │
│ gender          │      │ customer_key (FK)     │      │ brand            │
│ age_group       │      │ product_key (FK)      │      │ subcategory      │
│ city            │      │ category_key (FK)     │      │ launch_date      │
│ region          │      │ payment_key (FK)      │      │ stock_quantity   │
│registration_date│      │ shipping_key (FK)     │      └────────┬─────────┘
│customer_segment │      │ quantity              │               │
└─────────────────┘      │ gross_amount          │      ┌────────┴─────────┐
                         │ discount_amount       │      │  Dim_Category    │
┌─────────────────┐      │ net_amount            │      │──────────────────│
│  Dim_Payment    │      │ cost_amount           │─────►│category_key (PK) │
│─────────────────│      │ profit_amount         │      │ category_name    │
│payment_key (PK) │◄─────│                       │      │ parent_category  │
│ payment_method  │      └───────────┬───────────┘      │ seasonal_flag    │
└─────────────────┘                  │                   └──────────────────┘
                          ┌──────────┴────────┐
                          │  Dim_Shipping     │
                          │───────────────────│
                          │shipping_key (PK)  │
                          │ shipping_type     │
                          │ delivery_days     │
                          └───────────────────┘
```

**Schema Type:** Star Schema (1 Fact + 6 Dimensions)
**Grain:** One row per product line item in an order
**Date Range:** 2022-01-01 to 2025-12-31


### 1.2 DDL — CREATE TABLE Statements

In [ ]:
# ── 1.2 Schema DDL ──────────────────────────────────────────

conn.execute("""
CREATE TABLE Dim_Date (
    date_key       INTEGER PRIMARY KEY
    ,full_date     DATE NOT NULL
    ,day           INTEGER NOT NULL
    ,month         INTEGER NOT NULL
    ,month_name    VARCHAR NOT NULL
    ,quarter       INTEGER NOT NULL
    ,year          INTEGER NOT NULL
    ,week_number   INTEGER NOT NULL
);
""")

conn.execute("""
CREATE TABLE Dim_Customer (
    customer_key       INTEGER PRIMARY KEY
    ,customer_id       VARCHAR NOT NULL
    ,customer_name     VARCHAR NOT NULL
    ,gender            VARCHAR NOT NULL
    ,age_group         VARCHAR NOT NULL
    ,city              VARCHAR NOT NULL
    ,region            VARCHAR NOT NULL
    ,registration_date DATE NOT NULL
    ,customer_segment  VARCHAR NOT NULL
);
""")

conn.execute("""
CREATE TABLE Dim_Product (
    product_key     INTEGER PRIMARY KEY
    ,product_id     VARCHAR NOT NULL
    ,product_name   VARCHAR NOT NULL
    ,brand          VARCHAR NOT NULL
    ,subcategory    VARCHAR NOT NULL
    ,launch_date    DATE NOT NULL
    ,stock_quantity INTEGER NOT NULL
);
""")

conn.execute("""
CREATE TABLE Dim_Category (
    category_key     INTEGER PRIMARY KEY
    ,category_name   VARCHAR NOT NULL
    ,parent_category VARCHAR NOT NULL
    ,seasonal_flag   VARCHAR NOT NULL
);
""")

conn.execute("""
CREATE TABLE Dim_Payment (
    payment_key    INTEGER PRIMARY KEY
    ,payment_method VARCHAR NOT NULL
);
""")

conn.execute("""
CREATE TABLE Dim_Shipping (
    shipping_key   INTEGER PRIMARY KEY
    ,shipping_type VARCHAR NOT NULL
    ,delivery_days INTEGER NOT NULL
);
""")

conn.execute("""
CREATE TABLE Fact_Order_Line (
    order_line_id   INTEGER PRIMARY KEY
    ,order_id       INTEGER NOT NULL
    ,date_key       INTEGER NOT NULL REFERENCES Dim_Date(date_key)
    ,customer_key   INTEGER NOT NULL REFERENCES Dim_Customer(customer_key)
    ,product_key    INTEGER NOT NULL REFERENCES Dim_Product(product_key)
    ,category_key   INTEGER NOT NULL REFERENCES Dim_Category(category_key)
    ,payment_key    INTEGER NOT NULL REFERENCES Dim_Payment(payment_key)
    ,shipping_key   INTEGER NOT NULL REFERENCES Dim_Shipping(shipping_key)
    ,quantity       INTEGER NOT NULL
    ,gross_amount   DECIMAL(12,2) NOT NULL
    ,discount_amount DECIMAL(12,2) NOT NULL
    ,net_amount     DECIMAL(12,2) NOT NULL
    ,cost_amount    DECIMAL(12,2) NOT NULL
    ,profit_amount  DECIMAL(12,2) NOT NULL
);
""")

print("All 7 tables created successfully.")

---
## 2. Data Generation

**Seed = 42** for full reproducibility.

### Intentional Patterns Engineered Into the Data

| # | Pattern | How It's Built |
|---|---------|---------------|
| 1 | Q4 seasonality | Nov-Dec orders weighted 40% higher |
| 2 | Jan-Feb slowdown | Jan-Feb orders weighted 20% lower |
| 3 | Central region outperforms | Central customers get 1.15x profit margin |
| 4 | Premium segment 2x AOV | Premium gets higher gross_amount range |
| 5 | Products 1-3 declining | Last 3 months of 2025: amounts reduced 50-70% |
| 6 | 5 power users | customer_keys 1-5 assigned 25+ orders each |
| 7 | YoY growth | Volume increases each year |
| 8 | 200+ repeat customers | Ensures repeat purchase analysis works |
| 9 | Co-purchase pairings | 8 product pairs appear together ~30% of the time |


### 2.1 Dimension Tables

In [ ]:
# ── 2.1a  Dim_Date ──────────────────────────────────────────
import calendar

start_date = date(2022, 1, 1)
end_date   = date(2025, 12, 31)

date_rows = []
d = start_date
key = 1
while d <= end_date:
    date_rows.append({
        'date_key':    key
        ,'full_date':  d
        ,'day':        d.day
        ,'month':      d.month
        ,'month_name': calendar.month_name[d.month]
        ,'quarter':    (d.month - 1) // 3 + 1
        ,'year':       d.year
        ,'week_number': d.isocalendar()[1]
    })
    d += timedelta(days=1)
    key += 1

df_date = pd.DataFrame(date_rows)
print(f"Dim_Date: {len(df_date)} rows  |  {df_date['full_date'].min()} to {df_date['full_date'].max()}")
df_date.head()

In [ ]:
# ── 2.1b  Dim_Customer ──────────────────────────────────────

REGIONS = ['North', 'South', 'East', 'West', 'Central']
SEGMENTS = ['Regular', 'Premium', 'VIP', 'New']
AGE_GROUPS = ['18-24', '25-34', '35-44', '45-54', '55+']
GENDERS = ['Male', 'Female']

EGYPTIAN_CITIES = {
    'North':   ['Alexandria', 'Damietta', 'Kafr El Sheikh', 'Beheira'],
    'South':   ['Luxor', 'Aswan', 'Qena', 'Sohag'],
    'East':    ['Ismailia', 'Port Said', 'Suez', 'Red Sea'],
    'West':    ['Marsa Matrouh', '6th of October', 'Fayoum', 'Beni Suef'],
    'Central': ['Cairo', 'Giza', 'Qalyubia', 'Helwan']
}

# Segment distribution: Regular 40%, Premium 25%, VIP 10%, New 25%
segment_weights = [0.40, 0.25, 0.10, 0.25]

customer_rows = []
for i in range(1, 501):
    region  = random.choice(REGIONS)
    segment = random.choices(SEGMENTS, weights=segment_weights, k=1)[0]
    city    = random.choice(EGYPTIAN_CITIES[region])
    reg_date = fake.date_between(start_date=date(2020, 1, 1), end_date=date(2024, 12, 31))

    customer_rows.append({
        'customer_key':      i
        ,'customer_id':      f'CUST-{i:04d}'
        ,'customer_name':    fake.name()
        ,'gender':           random.choice(GENDERS)
        ,'age_group':        random.choice(AGE_GROUPS)
        ,'city':             city
        ,'region':           region
        ,'registration_date': reg_date
        ,'customer_segment': segment
    })

df_customer = pd.DataFrame(customer_rows)
print(f"Dim_Customer: {len(df_customer)} rows")
print(f"  Regions:  {df_customer['region'].value_counts().to_dict()}")
print(f"  Segments: {df_customer['customer_segment'].value_counts().to_dict()}")

In [ ]:
# ── 2.1c  Dim_Category ──────────────────────────────────────

category_data = [
    (1, 'Electronics',  'Technology',      'N')
    ,(2, 'Fashion',     'Lifestyle',       'Y')
    ,(3, 'Home',        'Lifestyle',       'N')
    ,(4, 'Beauty',      'Personal Care',   'Y')
    ,(5, 'Sports',      'Active Living',   'Y')
    ,(6, 'Books',       'Education',       'N')
    ,(7, 'Food',        'Essentials',      'Y')
    ,(8, 'Toys',        'Entertainment',   'Y')
]

df_category = pd.DataFrame(category_data,
    columns=['category_key', 'category_name', 'parent_category', 'seasonal_flag'])
print(f"Dim_Category: {len(df_category)} rows")
df_category

In [ ]:
# ── 2.1d  Dim_Product ───────────────────────────────────────

BRANDS = [
    'TechPro', 'StyleHub', 'HomeNest', 'GlowUp', 'FitZone'
    ,'BookWorm', 'FreshBite', 'PlayTime', 'SmartLife', 'EliteGear'
]

SUBCATEGORIES = {
    'Electronics':  ['Smartphones', 'Laptops', 'Headphones', 'Tablets', 'Cameras'],
    'Fashion':      ['Dresses', 'Shirts', 'Shoes', 'Handbags', 'Watches'],
    'Home':         ['Furniture', 'Kitchen', 'Lighting', 'Decor', 'Bedding'],
    'Beauty':       ['Skincare', 'Makeup', 'Haircare', 'Fragrances', 'Nail Care'],
    'Sports':       ['Running Shoes', 'Gym Equipment', 'Sportswear', 'Accessories', 'Yoga'],
    'Books':        ['Fiction', 'Non-Fiction', 'Academic', 'Comics', 'Self-Help'],
    'Food':         ['Snacks', 'Beverages', 'Organic', 'Supplements', 'Gourmet'],
    'Toys':         ['Board Games', 'Action Figures', 'Puzzles', 'Educational', 'Outdoor']
}

PRODUCT_NAMES = {
    'Smartphones': ['Galaxy Ultra', 'iPhone Pro Max', 'Pixel 8', 'OnePlus Nord'],
    'Laptops': ['ProBook 15', 'ThinkPad X1', 'MacBook Air M3', 'ZenBook 14'],
    'Headphones': ['AirPods Max', 'WH-1000XM5', 'Galaxy Buds', 'FreeBuds Pro'],
    'Tablets': ['iPad Air', 'Galaxy Tab S9', 'Surface Go', 'MatePad Pro'],
    'Cameras': ['EOS R6', 'Alpha A7', 'Z6 III', 'X-T5'],
    'Dresses': ['Summer Maxi', 'Evening Gown', 'Casual Wrap', 'Linen Shift'],
    'Shirts': ['Oxford Classic', 'Slim Fit Polo', 'Linen Casual', 'Denim Button'],
    'Shoes': ['Air Max 90', 'Ultra Boost', 'Classic Leather', 'Gel Kayano'],
    'Handbags': ['Tote Classic', 'Crossbody Mini', 'Clutch Evening', 'Backpack Pro'],
    'Watches': ['Classic Chrono', 'Smart Watch X', 'Diver Pro', 'Minimalist'],
    'Furniture': ['Ergonomic Chair', 'Standing Desk', 'Bookshelf Oak', 'Sofa 3-Seat'],
    'Kitchen': ['Coffee Maker Pro', 'Air Fryer XL', 'Blender Max', 'Knife Set'],
    'Lighting': ['LED Desk Lamp', 'Floor Lamp Arc', 'Smart Bulb Kit', 'Pendant Light'],
    'Decor': ['Wall Art Set', 'Throw Pillows', 'Ceramic Vase', 'Mirror Round'],
    'Bedding': ['Memory Foam Pillow', 'Cotton Sheet Set', 'Weighted Blanket', 'Duvet King'],
    'Skincare': ['Vitamin C Serum', 'Retinol Cream', 'Sunscreen SPF50', 'Clay Mask'],
    'Makeup': ['Foundation Pro', 'Lipstick Matte', 'Eyeshadow Palette', 'Mascara Volume'],
    'Haircare': ['Shampoo Argan', 'Conditioner Silk', 'Hair Oil Repair', 'Styling Gel'],
    'Fragrances': ['Eau de Parfum', 'Body Mist Fresh', 'Cologne Sport', 'Perfume Gift Set'],
    'Nail Care': ['Gel Polish Kit', 'Nail Strengthener', 'Cuticle Oil', 'Press-On Nails'],
    'Running Shoes': ['Trail Runner X', 'Speed Racer', 'Comfort Jog', 'Marathon Elite'],
    'Gym Equipment': ['Dumbbell Set', 'Resistance Bands', 'Pull-Up Bar', 'Kettlebell 20kg'],
    'Sportswear': ['Compression Tights', 'Dry-Fit Shirt', 'Track Jacket', 'Sports Bra'],
    'Accessories': ['Sports Socks 6pk', 'Wrist Bands', 'Water Bottle 1L', 'Gym Bag Pro'],
    'Yoga': ['Yoga Mat Premium', 'Yoga Block Set', 'Resistance Loop', 'Balance Ball'],
    'Fiction': ['Mystery Novel', 'Sci-Fi Epic', 'Romance Bestseller', 'Thriller Page-Turner'],
    'Non-Fiction': ['Business Strategy', 'History of Egypt', 'Science Explained', 'Memoir Life'],
    'Academic': ['Data Science 101', 'SQL Mastery', 'Python Cookbook', 'Statistics Intro'],
    'Comics': ['Manga Collection', 'Superhero Anthology', 'Graphic Novel Art', 'Comedy Strip'],
    'Self-Help': ['Atomic Habits', 'Deep Work Guide', 'Mindfulness Daily', 'Leadership Book'],
    'Snacks': ['Nut Mix Premium', 'Protein Bars 12pk', 'Dark Chocolate', 'Rice Cakes'],
    'Beverages': ['Green Tea Box', 'Coffee Beans 1kg', 'Sparkling Water', 'Juice Variety'],
    'Organic': ['Olive Oil Extra', 'Honey Raw', 'Quinoa Pack', 'Chia Seeds'],
    'Supplements': ['Vitamin D3', 'Omega Fish Oil', 'Probiotic Daily', 'Collagen Powder'],
    'Gourmet': ['Truffle Oil', 'Saffron Premium', 'Aged Cheese Set', 'Artisan Pasta'],
    'Board Games': ['Strategy Kingdom', 'Family Quiz Night', 'Card Game Mega', 'Puzzle Adventure'],
    'Action Figures': ['Hero Collection', 'Space Explorer', 'Dinosaur Set', 'Robot Warriors'],
    'Puzzles': ['1000pc Landscape', 'Wooden Brain Teaser', '3D Castle', 'Jigsaw World Map'],
    'Educational': ['STEM Kit Junior', 'Science Lab', 'Math Games', 'Language Cards'],
    'Outdoor': ['Swing Set', 'Water Gun Pack', 'Kite Deluxe', 'Sand Toys']
}

# Assign products: ~12-13 per category (100 total)
product_rows = []
product_key = 1
cat_names = [c[1] for c in category_data]
cat_keys  = {c[1]: c[0] for c in category_data}

# Build product-to-category mapping for fact table
product_category_map = {}

for cat_name in cat_names:
    subcats = SUBCATEGORIES[cat_name]
    # Assign 12-13 products per category (total ~100)
    products_per_cat = 13 if product_key <= 40 else 12
    if product_key > 100:
        break
    for j in range(products_per_cat):
        if product_key > 100:
            break
        subcat = subcats[j % len(subcats)]
        names_pool = PRODUCT_NAMES.get(subcat, [f'{subcat} Item {j+1}'])
        pname = names_pool[j % len(names_pool)]
        brand = BRANDS[(product_key - 1) % len(BRANDS)]
        launch = fake.date_between(start_date=date(2020, 1, 1), end_date=date(2023, 12, 31))
        stock = random.randint(10, 500)

        product_rows.append({
            'product_key':    product_key
            ,'product_id':    f'PROD-{product_key:04d}'
            ,'product_name':  pname
            ,'brand':         brand
            ,'subcategory':   subcat
            ,'launch_date':   launch
            ,'stock_quantity': stock
        })
        product_category_map[product_key] = cat_keys[cat_name]
        product_key += 1

df_product = pd.DataFrame(product_rows)
print(f"Dim_Product: {len(df_product)} rows across {df_product['brand'].nunique()} brands")
print(f"  Products per brand: {df_product['brand'].value_counts().to_dict()}")

In [ ]:
# ── 2.1e  Dim_Payment ───────────────────────────────────────

payment_data = [
    (1, 'Credit Card')
    ,(2, 'Debit Card')
    ,(3, 'Cash on Delivery')
    ,(4, 'Digital Wallet')
    ,(5, 'Bank Transfer')
]
df_payment = pd.DataFrame(payment_data, columns=['payment_key', 'payment_method'])
print(f"Dim_Payment: {len(df_payment)} rows")
df_payment

In [ ]:
# ── 2.1f  Dim_Shipping ──────────────────────────────────────

shipping_data = [
    (1, 'Standard',  6)
    ,(2, 'Express',   2)
    ,(3, 'Same Day',  0)
    ,(4, 'Free',      8)
]
df_shipping = pd.DataFrame(shipping_data, columns=['shipping_key', 'shipping_type', 'delivery_days'])
print(f"Dim_Shipping: {len(df_shipping)} rows")
df_shipping

### 2.2 Fact Table — Fact_Order_Line (15,000 rows)

In [ ]:
# ── 2.2  Fact_Order_Line ─────────────────────────────────────
# This is the most complex generation — 15,000 rows with intentional patterns

N_ROWS = 15000

# ── Date key lookup ──
date_lookup = df_date.set_index('full_date')['date_key'].to_dict()
all_dates   = list(date_lookup.keys())

# ── Customer lookup by segment ──
customer_segments = df_customer.set_index('customer_key')['customer_segment'].to_dict()
customer_regions  = df_customer.set_index('customer_key')['region'].to_dict()

# ── Monthly weights for seasonality ──
MONTH_WEIGHTS = {
    1: 0.80, 2: 0.80,   # Jan-Feb slowdown
    3: 1.00, 4: 1.00, 5: 1.00, 6: 1.00,
    7: 1.00, 8: 1.00, 9: 1.00, 10: 1.05,
    11: 1.40, 12: 1.40   # Nov-Dec Q4 spike
}

# ── Year weights for YoY growth ──
YEAR_WEIGHTS = {2022: 0.75, 2023: 0.90, 2024: 1.05, 2025: 1.30}

# ── Build weighted date pool ──
weighted_dates = []
for d_obj in all_dates:
    w = MONTH_WEIGHTS[d_obj.month] * YEAR_WEIGHTS[d_obj.year]
    weighted_dates.extend([d_obj] * int(w * 10))

# ── Co-purchase product pairs (8 pairings) ──
# Product keys mapped to actual generated products:
# Electronics: 1-13, Fashion: 14-26, Home: 27-39, Beauty: 40-52
# Sports: 53-64, Books: 65-76, Food: 77-88, Toys: 89-100
CO_PURCHASE_PAIRS = [
    (1, 3),    # Galaxy Ultra (smartphone) + AirPods Max (headphones)
    (2, 4),    # ProBook 15 (laptop) + iPad Air (tablet)
    (14, 17),  # Summer Maxi (dress) + Tote Classic (handbag)
    (28, 38),  # Coffee Maker Pro + Blender Max (kitchen appliances)
    (42, 47),  # Shampoo Argan + Conditioner Silk (haircare)
    (53, 56),  # Trail Runner X + Sports Socks 6pk (running gear)
    (67, 69),  # Data Science 101 + Atomic Habits (books)
    (78, 83),  # Green Tea Box + Coffee Beans 1kg (beverages)
]

# ── Power users: customer_keys 1-5 get 25+ orders each ──
POWER_USERS = [1, 2, 3, 4, 5]
POWER_USER_MIN_ORDERS = 25

# ── Generate orders ──
# Target: ~15,000 lines from ~6,500 orders (avg ~2.3 lines/order)
order_shells = []
order_id = 1

# First, generate guaranteed orders for power users
for ck in POWER_USERS:
    for _ in range(POWER_USER_MIN_ORDERS):
        order_date = random.choice(weighted_dates)
        order_shells.append({
            'order_id':     order_id
            ,'order_date':  order_date
            ,'customer_key': ck
            ,'payment_key':  random.randint(1, 5)
            ,'shipping_key': random.randint(1, 4)
        })
        order_id += 1

# Ensure 250+ customers have repeat purchases
repeat_customers = random.sample(range(6, 501), 250)
for ck in repeat_customers:
    n_orders = random.randint(2, 6)
    for _ in range(n_orders):
        order_date = random.choice(weighted_dates)
        order_shells.append({
            'order_id':     order_id
            ,'order_date':  order_date
            ,'customer_key': ck
            ,'payment_key':  random.randint(1, 5)
            ,'shipping_key': random.randint(1, 4)
        })
        order_id += 1

# Fill remaining orders to reach ~6500 total (to produce ~15K lines)
remaining_target = 6500 - len(order_shells)
all_customers = list(range(1, 501))
for _ in range(max(0, remaining_target)):
    ck = random.choice(all_customers)
    order_date = random.choice(weighted_dates)
    order_shells.append({
        'order_id':     order_id
        ,'order_date':  order_date
        ,'customer_key': ck
        ,'payment_key':  random.randint(1, 5)
        ,'shipping_key': random.randint(1, 4)
    })
    order_id += 1

random.shuffle(order_shells)

n_customers = len(set(o['customer_key'] for o in order_shells))
print(f"Generated {len(order_shells)} order shells")
print(f"Customers with orders: {n_customers}")

In [ ]:
# ── 2.2b  Generate line items with pricing logic ──────────

all_product_keys = list(range(1, 101))
co_purchase_dict = {}
for a, b in CO_PURCHASE_PAIRS:
    co_purchase_dict.setdefault(a, []).append(b)
    co_purchase_dict.setdefault(b, []).append(a)

fact_rows = []
line_id = 1

for shell in order_shells:
    if line_id > N_ROWS:
        break

    # 1-5 line items per order (weighted toward 2-3)
    n_lines = random.choices([1, 2, 3, 4, 5], weights=[15, 35, 28, 14, 8], k=1)[0]

    # Pick products for this order
    order_products = []
    first_product = random.choice(all_product_keys)
    order_products.append(first_product)

    # Co-purchase logic: 30% chance to add paired product
    if first_product in co_purchase_dict and random.random() < 0.30:
        paired = random.choice(co_purchase_dict[first_product])
        if paired not in order_products:
            order_products.append(paired)

    # Fill remaining line items with random products
    while len(order_products) < n_lines:
        p = random.choice(all_product_keys)
        if p not in order_products:
            order_products.append(p)

    order_products = order_products[:n_lines]

    for pk in order_products:
        if line_id > N_ROWS:
            break

        cat_key   = product_category_map[pk]
        segment   = customer_segments[shell['customer_key']]
        region    = customer_regions[shell['customer_key']]
        order_dt  = shell['order_date']

        # ── Pattern: Products 1-3 — skip most orders in Oct-Dec 2025 (decline) ──
        if pk in [1, 2, 3] and order_dt.year == 2025 and order_dt.month >= 10:
            if random.random() < 0.70:
                continue

        # ── Pricing logic ──
        if segment == 'Premium':
            gross = round(random.uniform(200, 5000), 2)
        elif segment == 'VIP':
            gross = round(random.uniform(300, 5000), 2)
        else:
            gross = round(random.uniform(50, 2000), 2)

        # Discount: 0-30%
        discount_pct = random.uniform(0, 0.30)
        discount_amt = round(gross * discount_pct, 2)
        net = round(gross - discount_amt, 2)

        # Cost: 40-70% of gross
        cost_pct = random.uniform(0.40, 0.70)
        cost = round(gross * cost_pct, 2)

        # ── Pattern: Central region outperforms by ~15% margin ──
        if region == 'Central':
            cost = round(cost * 0.85, 2)

        # ── Pattern: Products 1-3 price drop in Oct-Dec 2025 ──
        if pk in [1, 2, 3] and order_dt.year == 2025 and order_dt.month >= 10:
            gross = round(gross * 0.35, 2)
            discount_amt = round(gross * discount_pct, 2)
            net = round(gross - discount_amt, 2)
            cost = round(gross * cost_pct, 2)

        # ── Pattern: Electronics Q4 boost ──
        if cat_key == 1 and order_dt.month in [11, 12]:
            gross = round(gross * 1.25, 2)
            discount_amt = round(gross * discount_pct, 2)
            net = round(gross - discount_amt, 2)
            # cost stays the same -> higher margin in Q4 for electronics

        profit = round(net - cost, 2)
        qty = random.choices([1, 2, 3, 4, 5], weights=[50, 25, 15, 7, 3], k=1)[0]

        fact_rows.append({
            'order_line_id': line_id
            ,'order_id':     shell['order_id']
            ,'date_key':     date_lookup[order_dt]
            ,'customer_key': shell['customer_key']
            ,'product_key':  pk
            ,'category_key': cat_key
            ,'payment_key':  shell['payment_key']
            ,'shipping_key': shell['shipping_key']
            ,'quantity':     qty
            ,'gross_amount': gross
            ,'discount_amount': discount_amt
            ,'net_amount':   net
            ,'cost_amount':  cost
            ,'profit_amount': profit
        })
        line_id += 1

df_fact = pd.DataFrame(fact_rows)
print(f"Fact_Order_Line: {len(df_fact)} rows")
print(f"  Unique orders: {df_fact['order_id'].nunique()}")
date_min = df_date[df_date['date_key'].isin(df_fact['date_key'])]['full_date'].min()
date_max = df_date[df_date['date_key'].isin(df_fact['date_key'])]['full_date'].max()
print(f"  Date range: {date_min} to {date_max}")
print(f"  Revenue total: ${df_fact['net_amount'].sum():,.2f}")

---
## 3. Data Loading

Insert all generated DataFrames into DuckDB tables.


In [ ]:
# ── 3. Load DataFrames into DuckDB ──────────────────────────

conn.execute("INSERT INTO Dim_Date SELECT * FROM df_date")
conn.execute("INSERT INTO Dim_Customer SELECT * FROM df_customer")
conn.execute("INSERT INTO Dim_Product SELECT * FROM df_product")
conn.execute("INSERT INTO Dim_Category SELECT * FROM df_category")
conn.execute("INSERT INTO Dim_Payment SELECT * FROM df_payment")
conn.execute("INSERT INTO Dim_Shipping SELECT * FROM df_shipping")
conn.execute("INSERT INTO Fact_Order_Line SELECT * FROM df_fact")

# Verify row counts
for table in ['Dim_Date', 'Dim_Customer', 'Dim_Product', 'Dim_Category',
              'Dim_Payment', 'Dim_Shipping', 'Fact_Order_Line']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:25s} → {count:,} rows loaded")

print("\nAll data loaded successfully.")

---
## 4. Data Validation

Seven checks to prove data quality and confirm intentional patterns.


### 4.1 Row Counts

In [ ]:
# ── 4.1  Row Counts ─────────────────────────────────────────

expected = {
    'Dim_Date': 1461, 'Dim_Customer': 500, 'Dim_Product': 100,
    'Dim_Category': 8, 'Dim_Payment': 5, 'Dim_Shipping': 4,
    'Fact_Order_Line': 15000
}

rows = []
for table, exp in expected.items():
    actual = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    status = 'PASS' if actual >= exp * 0.95 else 'FAIL'  # Allow 5% tolerance for fact
    rows.append({'Table': table, 'Expected': exp, 'Actual': actual, 'Status': status})

df_counts = pd.DataFrame(rows)
print(df_counts.to_string(index=False))

### 4.2 NULL Foreign Key Check

In [ ]:
# ── 4.2  NULL FK Check ──────────────────────────────────────

fk_cols = ['date_key', 'customer_key', 'product_key',
           'category_key', 'payment_key', 'shipping_key']

null_rows = []
for col in fk_cols:
    nulls = conn.execute(f"SELECT COUNT(*) FROM Fact_Order_Line WHERE {col} IS NULL").fetchone()[0]
    null_rows.append({'FK Column': col, 'NULL Count': nulls, 'Status': 'PASS' if nulls == 0 else 'FAIL'})

df_nulls = pd.DataFrame(null_rows)
print(df_nulls.to_string(index=False))

### 4.3 Date Range Verification

In [ ]:
# ── 4.3  Date Range ─────────────────────────────────────────

result = conn.execute("""
SELECT
    MIN(d.full_date) AS min_date
    ,MAX(d.full_date) AS max_date
    ,COUNT(DISTINCT d.year) AS years_covered
FROM Fact_Order_Line f
JOIN Dim_Date d ON f.date_key = d.date_key
""").fetchdf()

print("Date Range in Fact Table:")
print(result.to_string(index=False))

### 4.4 Foreign Key Integrity (Orphan Check)

In [ ]:
# ── 4.4  FK Integrity ───────────────────────────────────────

fk_checks = [
    ('date_key',     'Dim_Date',     'date_key'),
    ('customer_key', 'Dim_Customer', 'customer_key'),
    ('product_key',  'Dim_Product',  'product_key'),
    ('category_key', 'Dim_Category', 'category_key'),
    ('payment_key',  'Dim_Payment',  'payment_key'),
    ('shipping_key', 'Dim_Shipping', 'shipping_key'),
]

orphan_rows = []
for fk_col, dim_table, dim_pk in fk_checks:
    orphans = conn.execute(f"""
        SELECT COUNT(*) FROM Fact_Order_Line f
        LEFT JOIN {dim_table} d ON f.{fk_col} = d.{dim_pk}
        WHERE d.{dim_pk} IS NULL
    """).fetchone()[0]
    orphan_rows.append({
        'FK': fk_col, 'Dimension': dim_table,
        'Orphan Rows': orphans, 'Status': 'PASS' if orphans == 0 else 'FAIL'
    })

df_orphans = pd.DataFrame(orphan_rows)
print(df_orphans.to_string(index=False))

### 4.5 Fact Table Descriptive Statistics

In [ ]:
# ── 4.5  Fact Table Stats ───────────────────────────────────

stats = conn.execute("""
SELECT
    'quantity'        AS measure ,MIN(quantity)        AS min_val ,ROUND(AVG(quantity),2)        AS avg_val ,MAX(quantity)        AS max_val ,ROUND(STDDEV(quantity),2)        AS stddev_val FROM Fact_Order_Line
UNION ALL SELECT
    'gross_amount'    ,MIN(gross_amount)    ,ROUND(AVG(gross_amount),2)    ,MAX(gross_amount)    ,ROUND(STDDEV(gross_amount),2)    FROM Fact_Order_Line
UNION ALL SELECT
    'discount_amount' ,MIN(discount_amount) ,ROUND(AVG(discount_amount),2) ,MAX(discount_amount) ,ROUND(STDDEV(discount_amount),2) FROM Fact_Order_Line
UNION ALL SELECT
    'net_amount'      ,MIN(net_amount)      ,ROUND(AVG(net_amount),2)      ,MAX(net_amount)      ,ROUND(STDDEV(net_amount),2)      FROM Fact_Order_Line
UNION ALL SELECT
    'cost_amount'     ,MIN(cost_amount)     ,ROUND(AVG(cost_amount),2)     ,MAX(cost_amount)     ,ROUND(STDDEV(cost_amount),2)     FROM Fact_Order_Line
UNION ALL SELECT
    'profit_amount'   ,MIN(profit_amount)   ,ROUND(AVG(profit_amount),2)   ,MAX(profit_amount)   ,ROUND(STDDEV(profit_amount),2)   FROM Fact_Order_Line
""").fetchdf()

print("Fact Table Descriptive Statistics:")
print(stats.to_string(index=False))

### 4.6 Seasonal Pattern Verification

In [ ]:
# ── 4.6  Seasonal Pattern ───────────────────────────────────

seasonal = conn.execute("""
SELECT
    d.month
    ,d.month_name
    ,COUNT(*) AS order_lines
    ,COUNT(DISTINCT f.order_id) AS unique_orders
    ,ROUND(SUM(f.net_amount), 2) AS total_revenue
FROM Fact_Order_Line f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.month, d.month_name
ORDER BY d.month
""").fetchdf()

print("Monthly Distribution (All Years Combined):")
print(seasonal.to_string(index=False))
print(f"\nNov-Dec avg orders: {seasonal[seasonal['month'].isin([11,12])]['unique_orders'].mean():,.0f}")
print(f"Jan-Feb avg orders: {seasonal[seasonal['month'].isin([1,2])]['unique_orders'].mean():,.0f}")
print(f"Other months avg:   {seasonal[~seasonal['month'].isin([1,2,11,12])]['unique_orders'].mean():,.0f}")

### 4.7 Year-over-Year Revenue Growth

In [ ]:
# ── 4.7  YoY Revenue ────────────────────────────────────────

yoy = conn.execute("""
SELECT
    d.year
    ,COUNT(DISTINCT f.order_id)    AS total_orders
    ,COUNT(*)                      AS total_lines
    ,ROUND(SUM(f.net_amount), 2)   AS total_revenue
    ,ROUND(SUM(f.profit_amount), 2) AS total_profit
FROM Fact_Order_Line f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
""").fetchdf()

print("Year-over-Year Performance:")
print(yoy.to_string(index=False))
print(f"\nRevenue Growth Confirmed: "
      f"{'YES' if yoy['total_revenue'].is_monotonic_increasing else 'NO — CHECK DATA!'}")

### 4.8 Bonus — Pattern Validation Summary

In [ ]:
# ── 4.8  Validate All Intentional Patterns ─────────────────

print("=" * 65)
print("INTENTIONAL PATTERN VALIDATION")
print("=" * 65)

# Pattern 1: Q4 seasonality (already shown in 4.6)
q4_rev = conn.execute("""
    SELECT ROUND(SUM(f.net_amount),2) FROM Fact_Order_Line f
    JOIN Dim_Date d ON f.date_key = d.date_key WHERE d.quarter = 4
""").fetchone()[0]
other_q_avg = conn.execute("""
    SELECT ROUND(AVG(q_rev),2) FROM (
        SELECT d.quarter, SUM(f.net_amount) AS q_rev FROM Fact_Order_Line f
        JOIN Dim_Date d ON f.date_key = d.date_key WHERE d.quarter != 4
        GROUP BY d.quarter
    )
""").fetchone()[0]
print(f"\n1. Q4 Seasonality: Q4 revenue = ${q4_rev:,.2f} vs other quarters avg = ${other_q_avg:,.2f}")

# Pattern 2: Central region outperforms
region_margin = conn.execute("""
    SELECT
        c.region
        ,ROUND(SUM(f.profit_amount) / SUM(f.net_amount) * 100, 2) AS profit_margin_pct
    FROM Fact_Order_Line f
    JOIN Dim_Customer c ON f.customer_key = c.customer_key
    GROUP BY c.region
    ORDER BY profit_margin_pct DESC
""").fetchdf()
print(f"\n2. Central Region Outperforms:")
print(region_margin.to_string(index=False))

# Pattern 3: Premium segment 2x AOV
segment_aov = conn.execute("""
    SELECT
        c.customer_segment
        ,ROUND(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2) AS aov
    FROM Fact_Order_Line f
    JOIN Dim_Customer c ON f.customer_key = c.customer_key
    GROUP BY c.customer_segment
    ORDER BY aov DESC
""").fetchdf()
print(f"\n3. Premium Segment AOV:")
print(segment_aov.to_string(index=False))

# Pattern 4: Products 1-3 declining in late 2025
decline = conn.execute("""
    SELECT
        f.product_key
        ,d.year
        ,d.month
        ,ROUND(SUM(f.net_amount), 2) AS revenue
    FROM Fact_Order_Line f
    JOIN Dim_Date d ON f.date_key = d.date_key
    WHERE f.product_key IN (1, 2, 3) AND d.year = 2025 AND d.month >= 7
    GROUP BY f.product_key, d.year, d.month
    ORDER BY f.product_key, d.month
""").fetchdf()
print(f"\n4. Products 1-3 Revenue in H2 2025 (should decline Oct-Dec):")
print(decline.to_string(index=False))

# Pattern 5: Power users
power = conn.execute("""
    SELECT customer_key, COUNT(DISTINCT order_id) AS order_count
    FROM Fact_Order_Line
    WHERE customer_key IN (1,2,3,4,5)
    GROUP BY customer_key
    ORDER BY customer_key
""").fetchdf()
print(f"\n5. Power Users (customer 1-5, target 25+ orders):")
print(power.to_string(index=False))

# Pattern 6: Repeat customers count
repeat = conn.execute("""
    SELECT COUNT(*) AS repeat_customers FROM (
        SELECT customer_key FROM Fact_Order_Line
        GROUP BY customer_key HAVING COUNT(DISTINCT order_id) >= 2
    )
""").fetchone()[0]
print(f"\n6. Repeat Customers (target 200+): {repeat}")

# Pattern 7: Co-purchase verification
print(f"\n7. Co-Purchase Pair Frequencies:")
for a, b in CO_PURCHASE_PAIRS[:4]:  # Show first 4 pairs
    co_count = conn.execute(f"""
        SELECT COUNT(DISTINCT f1.order_id)
        FROM Fact_Order_Line f1
        JOIN Fact_Order_Line f2 ON f1.order_id = f2.order_id
        WHERE f1.product_key = {a} AND f2.product_key = {b}
    """).fetchone()[0]
    total_a = conn.execute(f"""
        SELECT COUNT(DISTINCT order_id) FROM Fact_Order_Line WHERE product_key = {a}
    """).fetchone()[0]
    pname_a = df_product[df_product['product_key']==a]['product_name'].values[0]
    pname_b = df_product[df_product['product_key']==b]['product_name'].values[0]
    pct = round(co_count / max(total_a, 1) * 100, 1)
    print(f"   {pname_a} + {pname_b}: {co_count} co-purchases ({pct}% of product {a} orders)")

print(f"\n{'=' * 65}")
print("ALL PATTERNS VALIDATED")
print(f"{'=' * 65}")

---
---

# Future Sections (Sessions 2–5)

The following sections are placeholders to be completed in subsequent sessions.

---

## 5. Executive Summary

*To be completed in Session 5 — A compelling data-driven narrative summarizing all key findings.*


In [ ]:
# Executive Summary content will be added in Session 5

---
## 6. Methodology

*To be completed in Session 5 — Star schema rationale, analytical framework, and recommendation architecture.*


In [ ]:
# Methodology content will be added in Session 5

---
## 7. Core KPIs Dashboard

*To be completed in Session 2 — 7 core business KPIs with Plotly visualizations.*

| KPI | Description |
|-----|-------------|
| 1. Total Revenue Trend | Monthly revenue with 3-month moving average |
| 2. Gross Profit | Monthly profit with dual-axis revenue comparison |
| 3. Average Order Value | AOV by segment with reference line |
| 4. Customer Lifetime Value | Top spenders and distribution |
| 5. Repeat Purchase Rate | New vs returning customer trends |
| 6. Profit Margin by Category | Category profitability ranking |
| 7. Revenue Growth Rate (MoM) | Month-over-month with trend |


In [ ]:
# KPI Dashboard content will be added in Session 2

---
## 8. Section A: Time-Based Performance Analysis (Q1–Q6)

*To be completed in Session 3A*

| Q# | Topic |
|----|-------|
| Q1 | Cumulative Revenue |
| Q2 | Month-to-Date Performance |
| Q3 | Year-to-Date Profit |
| Q4 | 3-Month Moving Average |
| Q5 | Month-over-Month Comparison |
| Q6 | Revenue Acceleration / Deceleration |


In [ ]:
# Section A content will be added in Session 3

---
## 9. Section B: Ranking & Contribution Analysis (Q7–Q11)

*To be completed in Session 3B*

| Q# | Topic |
|----|-------|
| Q7 | Product Ranking by Category |
| Q8 | Product Contribution to Category Revenue |
| Q9 | Pareto Analysis (80/20 Rule) |
| Q10 | Region Profitability Ranking |
| Q11 | Brand Ranking Within Category |


In [ ]:
# Section B content will be added in Session 3

---
## 10. Section C: Customer Behavior Analytics (Q12–Q16)

*To be completed in Session 3C*

| Q# | Topic |
|----|-------|
| Q12 | Cumulative Customer Spending |
| Q13 | Inter-Purchase Time Analysis |
| Q14 | Customer Recency Ranking |
| Q15 | Spending Tiers (Quartiles) |
| Q16 | Top Percentile Analysis |


In [ ]:
# Section C content will be added in Session 3

---
## 11. Section D: Advanced Product & Category Analytics (Q17–Q21)

*To be completed in Session 3D*

| Q# | Topic |
|----|-------|
| Q17 | Revenue Volatility per Product |
| Q18 | Trending Categories |
| Q19 | Seasonality Analysis (YoY Same-Month) |
| Q20 | Profit Consistency Across Time |
| Q21 | Consecutive Revenue Decline Detection |


In [ ]:
# Section D content will be added in Session 3

---
## 12. Recommendation System

*To be completed in Session 4*

### Architecture:
```
Raw Data → SQL Scoring Engine → ML Enhancement → Hybrid Fusion → Top-4 Recommendations
```

### Components:
- **Stage 1:** SQL-Based Multi-Factor Scoring (6 normalized factors)
- **Stage 2:** ML Enhancement (Association Rules + Content-Based Filtering)
- **Stage 3:** Hybrid Scoring (0.55 SQL + 0.45 ML)
- **Stage 4:** Evaluation (Precision@4, Recall, Coverage, Diversity)


In [ ]:
# Recommendation System content will be added in Session 4

---
## 13. Key Findings & Business Recommendations

*To be completed in Session 5 — 7 strategic recommendations with evidence, actions, and priority.*


In [ ]:
# Key Findings content will be added in Session 5

---
## 14. Limitations & Future Work

*To be completed in Session 5*


In [ ]:
# Limitations content will be added in Session 5

---
## 15. Appendix: Data Dictionary

*To be completed in Session 5*


In [ ]:
# Data Dictionary content will be added in Session 5